# GPTQ + Marlin 최적화 버전

## 개요
vLLM의 Marlin 커널에 최적화된 GPTQ 양자화입니다.

### Marlin 커널이란?
- NVIDIA GPU용 초고속 4-bit 추론 커널
- 일반 GPTQ 대비 2~4배 빠름
- vLLM이 자동으로 적용

### Marlin 호환 조건
- W4A16 (4-bit weights, 16-bit activations)
- group_size = 128
- 표준 GPTQ 포맷

---

# 1. Import 및 환경 확인

In [1]:
import os
import sys
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print("=" * 60)
print("환경 정보")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU 없음 - CPU로 실행됩니다")
print("=" * 60)

환경 정보
Python: 3.10.13
PyTorch: 2.9.1
CUDA 사용 가능: False
⚠️  GPU 없음 - CPU로 실행됩니다


# 2. 하이퍼파라미터 설정

### Marlin 최적화 포인트
1. `group_size=128` (Marlin 필수)
2. 높은 캘리브레이션 품질 (성능 유지)
3. `actorder="weight"` (정확도 향상)

In [2]:
# ============================================================================
# 모델 설정
# ============================================================================
# 로컬 모델 경로 (다운로드 불필요!)
MODEL_ID = "./open/base_model"
OUT_DIR = "./model"  # 제출용 폴더명

# 데이터셋 설정
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# 캘리브레이션 설정 (품질 우선)
# ============================================================================
# Marlin은 속도를 올려주므로, 성능(PerfNorm) 유지가 핵심!
if torch.cuda.is_available():
    NUM_CALIBRATION_SAMPLES = 512
    MAX_SEQUENCE_LENGTH = 1024
else:
    NUM_CALIBRATION_SAMPLES = 256  # CPU에서도 품질 유지
    MAX_SEQUENCE_LENGTH = 512

# ============================================================================
# Marlin 호환 GPTQ 설정
# ============================================================================
SCHEME = "W4A16"          # 4-bit weights, 16-bit activations
GROUP_SIZE = 128          # Marlin 필수 조건!
ACTORDER = "weight"       # 정확도 향상 (weight 기반 순서)
DAMPENING_FRAC = 0.01     # Hessian 안정화

TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]  # 임베딩 레이어 제외

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("GPTQ + Marlin 최적화 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"SCHEME: {SCHEME}")
print(f"GROUP_SIZE: {GROUP_SIZE} (Marlin 호환)")
print(f"ACTORDER: {ACTORDER}")
print(f"NUM_CALIBRATION_SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"MAX_SEQUENCE_LENGTH: {MAX_SEQUENCE_LENGTH}")
print("=" * 60)

GPTQ + Marlin 최적화 설정
MODEL_ID: ./open/base_model
OUT_DIR: ./model
SCHEME: W4A16
GROUP_SIZE: 128 (Marlin 호환)
ACTORDER: weight
NUM_CALIBRATION_SAMPLES: 256
MAX_SEQUENCE_LENGTH: 512


# 3. 모델 로드

In [3]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

device_map = "auto" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32 if not torch.cuda.is_available() else torch.bfloat16,
    trust_remote_code=True,
    device_map=device_map,
)

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print(f"[INFO] 디바이스: {device_map}")
print("[INFO] 모델 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 디바이스: cpu
[INFO] 모델 로드 완료


# 4. 캘리브레이션 데이터셋

In [4]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터셋 크기: 256
[INFO] 데이터 전처리 완료


# 5. GPTQ 양자화 (Marlin 호환)

In [5]:
print("[INFO] GPTQ 양자화 시작 (Marlin 호환)")
print(f"  - scheme: {SCHEME}")
print(f"  - group_size: {GROUP_SIZE} (Marlin 필수)")
print(f"  - actorder: {ACTORDER}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드: 15-30분 예상\n")
else:
    print("\n⏳ CPU 모드: 2-4시간 예상\n")

# ============================================================================
# Marlin 호환 GPTQ 레시피
# ============================================================================
recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=GROUP_SIZE,      # Marlin 호환: 128
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,          # weight 기반 정렬
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] GPTQ 양자화 완료!")
print("[INFO] vLLM에서 자동으로 Marlin 커널 적용됨")

[INFO] GPTQ 양자화 시작 (Marlin 호환)
  - scheme: W4A16
  - group_size: 128 (Marlin 필수)
  - actorder: weight
  - samples: 256

⏳ CPU 모드: 2-4시간 예상



Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-09T22:41:14.225875+0900 | reset | INFO - Compression lifecycle reset
2026-02-09T22:41:14.226989+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-09T22:41:14.245392+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-09T22:41:14.245755+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-09T22:41:14.250961+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0209 22:41:14.272000 71243 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:36<00:00,  6.98it/s]

2026-02-09T22:41:51.118714+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

(30/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:37<00:00,  6.88it/s]

2026-02-09T23:05:29.025362+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 256 samples


2026-02-09T23:05:29.313024+0900 | compress | METRIC - time 0.29s
2026-02-09T23:05:29.313369+0900 | compress | METRIC - error 3258.94
2026-02-09T23:05:29.314195+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-09T23:05:29.314393+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-09T23:05:29.315786+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 256 samples
2026-02-09T23:05:29.499015+0900 | compress | METRIC - time 0.18s
2026-02-09T23:05:29.499354+0900 | compress | METRIC - error 923.58
2026-02-09T23:05:29.500203+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-09T23:05:29.500419+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-09T23:05:29.501072+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 256 samples
2026-02-09T23:05:29.683879+0900 | compress | METRIC - time 0.18s
2026-02-09T23:0

(31/31): Propagating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:00<00:00, 4515.31it/s]

2026-02-09T23:05:40.116133+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-09T23:05:40.121411+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] GPTQ 양자화 완료!
[INFO] vLLM에서 자동으로 Marlin 커널 적용됨


# 6. 모델 저장 및 크기 비교

In [8]:
print("[INFO] 모델 저장 중...")

# 기존 폴더 삭제 후 저장
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR
            , exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 파일 확인
print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  양자화 모델:   {quantized_size_gb:.2f} GB")
print(f"  ----------------------------------------")
print(f"  크기 감소:     {ORIGINAL_MODEL_SIZE_GB - quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print(f"  압축 배수:     {ORIGINAL_MODEL_SIZE_GB / quantized_size_gb:.2f}x")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-10T00:38:35.879522+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:00, 1807.51it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1407.7 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  양자화 모델:   1.42 GB
  ----------------------------------------
  크기 감소:     1.14 GB
  압축률:        55.4%
  압축 배수:     1.80x


# 7. 제출 파일 생성

In [9]:
zip_name = "submit_marlin"
print(f"[INFO] {zip_name}.zip 생성 중...")

# 기존 zip 삭제
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,  # "model" 폴더
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족 (≤ 10GB)")
else:
    print("❌ 용량 초과!")

print(f"\n📁 파일 위치: {os.path.abspath(f'{zip_name}.zip')}")
print("\n" + "=" * 60)
print("제출 파일 구조 확인")
print("=" * 60)
print(f"{zip_name}.zip")
print(f"└── model/")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"    ├── {f}")
print("=" * 60)

[INFO] submit_marlin.zip 생성 중...
[INFO] 생성 완료: submit_marlin.zip (0.88 GB)
✅ 용량 제한 충족 (≤ 10GB)

📁 파일 위치: /Users/imdonghyeon/Desktop/lg-aimers8-llm-compression/submit_marlin.zip

제출 파일 구조 확인
submit_marlin.zip
└── model/
    ├── chat_template.jinja
    ├── config.json
    ├── generation_config.json
    ├── merges.txt
    ├── model.safetensors
    ├── recipe.yaml
    ├── special_tokens_map.json
    ├── tokenizer.json
    ├── tokenizer_config.json
    ├── vocab.json


# 8. 예상 성능

### vLLM + Marlin 자동 적용

```
평가 서버 (L4 GPU + vLLM)
         ↓
모델 로드 (GPTQ W4A16, group_size=128)
         ↓
Marlin 커널 자동 감지 및 적용
         ↓
고속 추론 (2~4배 빠름)
```

### 예상 점수

| 항목 | 베이스라인 | Marlin 최적화 |
|------|-----------|---------------|
| PerfNorm | 0.95 | 0.95 (유지) |
| SpeedNorm | 0.30 | 0.50~0.65 |
| **Score** | **0.625** | **0.725~0.80** |

---

## 베이스라인과 차이점

| 항목 | 베이스라인 | Marlin 최적화 |
|------|-----------|---------------|
| group_size | 128 | 128 (동일) |
| actorder | static | **weight** |
| calibration | 256 | 256 (동일) |
| 폴더명 | model_xxx | **model** |

### 핵심 변경
1. `actorder="weight"`: 중요도 기반 가중치 정렬로 정확도 향상
2. `OUT_DIR="./model"`: 제출 규격에 맞는 폴더명

---